## MultiAgent Deep-research Workflow
This notebook creates a multi-agent AI research pipeline using multiple agents coordinated by a central Research Manager.

In [ ]:
import os
from agents import Agent, Runner, trace, gen_trace_id
from pydantic import BaseModel, Field

from dotenv import load_dotenv

# Change working directory to incorporate week 2 project.
notebook_dir = os.getcwd()
week2_dir = os.path.join(notebook_dir, '..', '..')
os.chdir(week2_dir)
print(f"Working directory: {os.getcwd()}")

### Clarifying Agent

Ask clarifying questions associated with the query.

In [ ]:
CLARIFYING_AGENT_MODEL = "gpt-4o-mini"

class ClarifyingQuestions(BaseModel):
    questions: list[str] = Field(
        description="Exactly 3 clarifying questions to better understand the user's research needs.",
        min_items=3,  
        max_items=3   
    )

CLARIFYING_AGENT_INSTRUCTIONS = "You are a helpful assistant that anayses the user's query and asks clarifying questions to help the user refine their query.\
    Analyse the user's query and send back 3 questions to help them refine their query. \
    Once you get these questions, please generate a comprehensive research query that will be used to search the web for relevant information."

clarifying_agent = Agent(
    name="Clarifying Agent",
    instructions=CLARIFYING_AGENT_INSTRUCTIONS,
    model=CLARIFYING_AGENT_MODEL,
    output_type=ClarifyingQuestions
)

### Evaluator Agent

Evaluates whether the answer generated so far for our query is good enough and meets the user's needs.

In [ ]:
class Evaluation(BaseModel):
    is_acceptable: bool
    feedback: str

EVALUATOR_AGENT_MODEL = "gpt-4.1-mini-2025-04-14"

EVALUATOR_AGENT_INSTRUCTIONS = "You are an evaluator that analyses the research report generated by the writer agent and determines whether it is of good quality.\
    You will be provided with the research report and the user's query. \
    You will need to determine whether the research report is of good quality and whether it satisfies the user's query. \
    You will also need to provide feedback on the research report and the user's query."

evaluator_agent = Agent(
    name="Evaluator Agent",
    instructions=EVALUATOR_AGENT_INSTRUCTIONS,
    model=EVALUATOR_AGENT_MODEL,
    output_type=Evaluation)

### Tools and Handoffs
Set up the tools and handoffs for the research manager.

In [ ]:
from deep_research.planner_agent import planner_agent
from deep_research.search_agent import search_agent
from deep_research.writer_agent import writer_agent
from deep_research.email_agent import email_agent

# Tools
clarification_tool = clarifying_agent.as_tool(
    tool_name="clarification_tool", 
    tool_description="Ask clarifying questions to the user to better understand and refine their research needs.")

planner_tool = planner_agent.as_tool(
    tool_name="planner_tool", 
    tool_description="Plan a list of web searches to perform based on the user's research needs.")

search_tool = search_agent.as_tool(
    tool_name="search_tool", 
    tool_description="Search the web for relevant information based on the user's research needs.")

writer_tool = writer_agent.as_tool(
    tool_name="writer_tool", 
    tool_description="Write a report on the findings of the web searches.")

evaluation_tool = evaluator_agent.as_tool(
    tool_name="evaluation_tool", 
    tool_description="Evaluate the quality of the research report.")

tools = [clarification_tool, planner_tool, search_tool, writer_tool, evaluation_tool]

# Handoffs
handoffs = [email_agent]

### Research Manager Agent
It should be autonomous with the ability to decide whether it needs to do more research. It is equiped with tools that will assist to perform the variuos tasks.

In [ ]:
RESEARCH_MANAGER_MODEL = "gpt-4o-mini"

RESEARCH_MANAGER_INSTRUCTIONS = """You are a research manager that oversees the research process of the query sent by the user. 
You have access to a team of agents that can help you with the research. 
You are responsible for deciding whether the research is complete and the answer is good enough. 
You can use the agents as tools to help you with the research. 

Please follow the following steps carefully for a successful and efficient research process:
1. Use the clarifying tool to formulate exactly 3 clarifying questions to better understand the user's research needs. You will then
send back the answers from the userto the clarifying agent/tool to formulate a comprehensive research query.
2. Take the query formulated by the clarifying agent and hand it over to the planner agent to come up with a list of web searches to perform.
3. Use the planner agent's output to search the web for relevant information using the search agent.
4. Once the search agent gives their output, you shall hand it over to the writer agent to write a report on the findings. 
5. Once the writer agent gives their output, you shall use the evaluator agent to evaluate the research.
6. If the evaluator agent deems the research to be of good quality, you shall hand-off the final report to the emailer tool to send the report to the user.
7. If the evaluator agent deems the research to be of poor quality, you shall repeat the process from step 3.

Crucial Rules:
- Ask for clarification from the user only once.
- Once you get the answers from the user, proceed to the next step.
- Only respond to the user once the research is complete.
"""

research_manager = Agent(
    name="Research Manager",
    instructions=RESEARCH_MANAGER_INSTRUCTIONS,
    tools=tools,
    handoffs=handoffs,
    model=RESEARCH_MANAGER_MODEL)

In [ ]:
import gradio as gr

async def run_research(message, history):
    # Format the history into the message structure required by OpenAI
    messages = []
    for human, bot in history:
        messages.append({"role": "user", "content": human})
        messages.append({"role": "assistant", "content": bot})
    messages.append({"role": "user", "content": message})

    # Run agent and return response
    trace_id = gen_trace_id()
    with trace("Research trace", trace_id=trace_id):
        print(f"View trace: https://platform.openai.com/traces/trace?trace_id={trace_id}")
        print("Starting research...")
        result = await Runner.run(research_manager, f"Query: {messages}")
        yield result.final_output

demo = gr.ChatInterface(run_research)
demo.launch()